In [1]:
import sys
import pandas as pd
import numpy as np
import re
import matplotlib
import nltk

print("=" * 45)
print("  ENVIRONMENT VERIFICATION")
print("=" * 45)
print(f"Python     : {sys.version[:6]}")
print(f"Pandas     : {pd.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Matplotlib : {matplotlib.__version__}")
print(f"NLTK       : {nltk.__version__}")
print("=" * 45)
print("🎉 Ready to build!")

  ENVIRONMENT VERIFICATION
Python     : 3.11.4
Pandas     : 2.1.0
NumPy      : 1.24.3
Matplotlib : 3.7.2
NLTK       : 3.8.1
🎉 Ready to build!


In [3]:
# ============================================
# CELL 2: Look at our raw data first
# Before writing ANY pattern, understand 
# what we are actually dealing with
# ============================================

# Read the sample file
file_path = "../data/sample/sample_chat.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"\nEach line with its LINE NUMBER:")
print("-" * 60)

for i, line in enumerate(lines, 1):
    print(f"Line {i:2d}: {repr(line)}")

Total lines: 26

Each line with its LINE NUMBER:
------------------------------------------------------------
Line  1: '3/14/23, 9:42 AM - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.\n'
Line  2: '3/14/23, 9:43 AM - Alice: Hey everyone! Good morning 😊\n'
Line  3: '3/14/23, 9:44 AM - Bob: Morning! How are you?\n'
Line  4: "3/14/23, 9:44 AM - Alice: I'm doing great! Just finished my assignment\n"
Line  5: '3/14/23, 9:45 AM - Bob: <Media omitted>\n'
Line  6: '3/14/23, 9:46 AM - Charlie: This message was deleted\n'
Line  7: '3/14/23, 9:47 AM - Alice: Did you all see the project deadline?\n'
Line  8: "3/14/23, 9:47 AM - Alice: It's been moved to next Friday\n"
Line  9: 'which gives us more time thankfully\n'
Line 10: "3/14/23, 9:48 AM - Bob: Oh that's great news! 🎉🎉\n"
Line 11: '3/14/23, 9:50 AM - Charlie: https://github.com/project-repo\n'
Line 12: "3/14/23, 10:00 AM - Dave joined using this grou

In [1]:
# ============================================
# CELL 3: The Core Question
# ============================================

import re

new_message_line  = "3/14/23, 9:42 AM - Alice: Hello everyone!"
continuation_line = "which gives us more time thankfully"
system_line       = "3/14/23, 10:00 AM - Dave joined using this group's invite link"
pm_line           = "3/16/23, 7:30 PM - Dave: Working late tonight"

print("WHAT DOES EACH LINE START WITH?")
print("-" * 45)

lines_to_check = [
    (new_message_line,  "Normal message"),
    (continuation_line, "Continuation"),
    (system_line,       "System/Group event"),
    (pm_line,           "PM message"),
]

for line, label in lines_to_check:
    first_char = line[0]
    is_digit   = first_char.isdigit()
    print(f"\n{label}:")
    print(f"  First character : '{first_char}'")
    print(f"  Is a digit?     : {is_digit}")
    print(f"  Line preview    : {line[:35]}...")

print()
print("=" * 45)
print("KEY INSIGHT:")
print("Every new message starts with a DIGIT")
print("Continuation lines start with a LETTER")
print("This is how we separate them!")
print("=" * 45)

WHAT DOES EACH LINE START WITH?
---------------------------------------------

Normal message:
  First character : '3'
  Is a digit?     : True
  Line preview    : 3/14/23, 9:42 AM - Alice: Hello eve...

Continuation:
  First character : 'w'
  Is a digit?     : False
  Line preview    : which gives us more time thankfully...

System/Group event:
  First character : '3'
  Is a digit?     : True
  Line preview    : 3/14/23, 10:00 AM - Dave joined usi...

PM message:
  First character : '3'
  Is a digit?     : True
  Line preview    : 3/16/23, 7:30 PM - Dave: Working la...

KEY INSIGHT:
Every new message starts with a DIGIT
Continuation lines start with a LETTER
This is how we separate them!


In [2]:
# ============================================
# CELL 4: Build the Regex Pattern
# One piece at a time — understand each symbol
# ============================================

import re

# We test against these two lines
sample_am = "3/14/23, 9:42 AM - Alice: Hello everyone!"
sample_pm = "3/16/23, 7:30 PM - Dave: Working late tonight"

print("BUILDING REGEX PATTERN PIECE BY PIECE")
print("=" * 55)

# ── PIECE 1: Month ───────────────────────────────────
p1 = r"\d{1,2}"
print(f"\n[1] Pattern : {p1}")
print(f"    Meaning  : 1 or 2 digits (month: 3 or 03 or 12)")
print(f"    Matches  : {re.findall(p1, sample_am[:2])}")

# ── PIECE 2: Month / Day ─────────────────────────────
p2 = r"\d{1,2}/\d{1,2}"
print(f"\n[2] Pattern : {p2}")
print(f"    Meaning  : month SLASH day")
print(f"    Matches  : {re.findall(p2, sample_am)}")

# ── PIECE 3: Month / Day / Year ──────────────────────
p3 = r"\d{1,2}/\d{1,2}/\d{2,4}"
print(f"\n[3] Pattern : {p3}")
print(f"    Meaning  : month/day/year (year = 23 or 2023)")
print(f"    Matches  : {re.findall(p3, sample_am)}")

# ── PIECE 4: Date , Space ────────────────────────────
p4 = r"\d{1,2}/\d{1,2}/\d{2,4},\s"
print(f"\n[4] Pattern : {p4}")
print(f"    Meaning  : date then COMMA then SPACE")
print(f"    Note     : \\s = any whitespace character")
print(f"    Matches  : {re.findall(p4, sample_am)}")

# ── PIECE 5: Date , Time ─────────────────────────────
p5 = r"\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}"
print(f"\n[5] Pattern : {p5}")
print(f"    Meaning  : date + time (hour:minute)")
print(f"    Note     : hour is 1-2 digits, minute always 2")
print(f"    Matches  : {re.findall(p5, sample_am)}")

# ── PIECE 6: Date , Time AM/PM ───────────────────────
p6 = r"\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s[AaPp][Mm]"
print(f"\n[6] Pattern : {p6}")
print(f"    Meaning  : date + time + AM or PM")
print(f"    Note     : [AaPp] matches A or a or P or p")
print(f"               [Mm]   matches M or m")
print(f"    AM test  : {re.findall(p6, sample_am)}")
print(f"    PM test  : {re.findall(p6, sample_pm)}")

# ── PIECE 7: Full timestamp + separator ──────────────
p7 = r"\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s[AaPp][Mm]\s-\s"
print(f"\n[7] Pattern : {p7}")
print(f"    Meaning  : full timestamp + ' - ' separator")
print(f"    This ' - ' is what separates TIME from MESSAGE")
print(f"    AM test  : {re.findall(p7, sample_am)}")
print(f"    PM test  : {re.findall(p7, sample_pm)}")

print()
print("=" * 55)
print(f"FINAL PATTERN:")
print(f"{p7}")
print("=" * 55)

BUILDING REGEX PATTERN PIECE BY PIECE

[1] Pattern : \d{1,2}
    Meaning  : 1 or 2 digits (month: 3 or 03 or 12)
    Matches  : ['3']

[2] Pattern : \d{1,2}/\d{1,2}
    Meaning  : month SLASH day
    Matches  : ['3/14']

[3] Pattern : \d{1,2}/\d{1,2}/\d{2,4}
    Meaning  : month/day/year (year = 23 or 2023)
    Matches  : ['3/14/23']

[4] Pattern : \d{1,2}/\d{1,2}/\d{2,4},\s
    Meaning  : date then COMMA then SPACE
    Note     : \s = any whitespace character
    Matches  : ['3/14/23, ']

[5] Pattern : \d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}
    Meaning  : date + time (hour:minute)
    Note     : hour is 1-2 digits, minute always 2
    Matches  : ['3/14/23, 9:42']

[6] Pattern : \d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s[AaPp][Mm]
    Meaning  : date + time + AM or PM
    Note     : [AaPp] matches A or a or P or p
               [Mm]   matches M or m
    AM test  : ['3/14/23, 9:42 AM']
    PM test  : ['3/16/23, 7:30 PM']

[7] Pattern : \d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s[AaPp][Mm

In [3]:
# ============================================
# CELL 5: Test pattern against ALL line types
# ============================================

import re

PATTERN = r"\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}\s[AaPp][Mm]\s-\s"

test_cases = [
    ("3/14/23, 9:42 AM - Alice: Hello!",                    True,  "Normal message AM"),
    ("3/16/23, 7:30 PM - Dave: Working late",               True,  "Normal message PM"),
    ("3/14/23, 9:42 AM - Messages are encrypted.",          True,  "System message"),
    ("3/14/23, 9:45 AM - Bob: <Media omitted>",             True,  "Media message"),
    ("3/14/23, 9:46 AM - Charlie: This message was deleted",True,  "Deleted message"),
    ("3/14/23, 10:00 AM - Dave joined the group",           True,  "Group event"),
    ("03/14/2023, 09:42 AM - Alice: Hi",                    True,  "4-digit year"),
    ("which gives us more time thankfully",                  False, "Continuation line"),
    ("Hello there this is random text",                      False, "Random text"),
    ("",                                                     False, "Empty line"),
]

print(f"{'DESCRIPTION':<35} {'EXPECTED':<10} {'GOT':<10} {'STATUS'}")
print("-" * 70)

score = 0
for line, expected, description in test_cases:
    match  = bool(re.match(PATTERN, line))
    passed = match == expected
    if passed:
        score += 1
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{description:<35} {str(expected):<10} {str(match):<10} {status}")

print("-" * 70)
print(f"\nScore: {score}/{len(test_cases)}")

if score == len(test_cases):
    print("🎉 Pattern handles ALL WhatsApp line types!")
    print("✅ Ready to build the full parser!")
else:
    print("⚠️ Some cases failed — let's debug together")

DESCRIPTION                         EXPECTED   GOT        STATUS
----------------------------------------------------------------------
Normal message AM                   True       True       ✅ PASS
Normal message PM                   True       True       ✅ PASS
System message                      True       True       ✅ PASS
Media message                       True       True       ✅ PASS
Deleted message                     True       True       ✅ PASS
Group event                         True       True       ✅ PASS
4-digit year                        True       True       ✅ PASS
Continuation line                   False      False      ✅ PASS
Random text                         False      False      ✅ PASS
Empty line                          False      False      ✅ PASS
----------------------------------------------------------------------

Score: 10/10
🎉 Pattern handles ALL WhatsApp line types!
✅ Ready to build the full parser!


In [4]:
# ============================================
# CELL 6: Extract information from ONE line
# Before parsing ALL lines, master ONE line
# ============================================

import re

# Our pattern - but now with GROUPS ( ) to capture parts
# Groups let us extract specific pieces of the match

PATTERN = r"(\d{1,2}/\d{1,2}/\d{2,4}),\s(\d{1,2}:\d{2}\s[AaPp][Mm])\s-\s(.*)"

# Notice 3 groups:
# Group 1 → (\d{1,2}/\d{1,2}/\d{2,4})  → captures DATE
# Group 2 → (\d{1,2}:\d{2}\s[AaPp][Mm]) → captures TIME  
# Group 3 → (.*)                          → captures EVERYTHING after " - "

test_lines = [
    "3/14/23, 9:42 AM - Alice: Hello everyone!",
    "3/14/23, 9:45 AM - Bob: <Media omitted>",
    "3/14/23, 10:00 AM - Dave joined using this group's invite link",
    "3/16/23, 7:30 PM - Dave: Working late tonight 😴",
]

print("EXTRACTING PARTS FROM EACH LINE")
print("=" * 55)

for line in test_lines:
    match = re.match(PATTERN, line)
    if match:
        date    = match.group(1)  # First group
        time    = match.group(2)  # Second group
        content = match.group(3)  # Third group - sender + message
        
        print(f"\nLine    : {line[:45]}...")
        print(f"Date    : {date}")
        print(f"Time    : {time}")
        print(f"Content : {content}")
        print(f"---")

EXTRACTING PARTS FROM EACH LINE

Line    : 3/14/23, 9:42 AM - Alice: Hello everyone!...
Date    : 3/14/23
Time    : 9:42 AM
Content : Alice: Hello everyone!
---

Line    : 3/14/23, 9:45 AM - Bob: <Media omitted>...
Date    : 3/14/23
Time    : 9:45 AM
Content : Bob: <Media omitted>
---

Line    : 3/14/23, 10:00 AM - Dave joined using this gr...
Date    : 3/14/23
Time    : 10:00 AM
Content : Dave joined using this group's invite link
---

Line    : 3/16/23, 7:30 PM - Dave: Working late tonight...
Date    : 3/16/23
Time    : 7:30 PM
Content : Dave: Working late tonight 😴
---


In [5]:
# ============================================
# CELL 7: Split "sender: message" into parts
# This is trickier than it looks
# ============================================

import re

contents = [
    "Alice: Hello everyone! Good morning 😊",
    "Bob: <Media omitted>",
    "Charlie: This message was deleted",
    "Dave joined using this group's invite link",     # No colon pattern
    "Messages and calls are end-to-end encrypted.",   # System message
    "Alice: Check this link https://github.com/test", # Colon in URL too!
]

print("SPLITTING SENDER FROM MESSAGE")
print("=" * 55)

for content in contents:
    
    # Check if this looks like "Name: message"
    # A sender name is typically:
    # - Starts at beginning
    # - Contains word characters
    # - Followed by ": "
    
    # split on FIRST colon only → maxsplit=1
    parts = content.split(":", 1)
    
    if len(parts) == 2:
        sender  = parts[0].strip()
        message = parts[1].strip()
    else:
        # No colon found → system/group message
        sender  = "System"
        message = content.strip()
    
    print(f"\nContent : {content[:50]}")
    print(f"Sender  : {sender}")
    print(f"Message : {message[:40]}")

SPLITTING SENDER FROM MESSAGE

Content : Alice: Hello everyone! Good morning 😊
Sender  : Alice
Message : Hello everyone! Good morning 😊

Content : Bob: <Media omitted>
Sender  : Bob
Message : <Media omitted>

Content : Charlie: This message was deleted
Sender  : Charlie
Message : This message was deleted

Content : Dave joined using this group's invite link
Sender  : System
Message : Dave joined using this group's invite li

Content : Messages and calls are end-to-end encrypted.
Sender  : System
Message : Messages and calls are end-to-end encryp

Content : Alice: Check this link https://github.com/test
Sender  : Alice
Message : Check this link https://github.com/test


In [6]:
# ============================================
# CELL 8: Classify each message into a type
# Why? Because media/deleted/system messages
# must be handled differently in analysis
# ============================================

def get_message_type(sender, message):
    """
    Classify a message into its type.
    
    Returns one of:
    - 'normal'    → regular text message
    - 'media'     → image, video, audio shared
    - 'deleted'   → message was deleted
    - 'system'    → WhatsApp system notification
    - 'group_event' → someone joined/left/was added
    """
    
    # Check sender first
    if sender == "System":
        # Could be system message or group event
        group_keywords = ["joined", "left", "added", 
                         "removed", "changed", "created"]
        if any(keyword in message.lower() for keyword in group_keywords):
            return "group_event"
        else:
            return "system"
    
    # Check message content
    if "<Media omitted>" in message:
        return "media"
    
    if "This message was deleted" in message:
        return "deleted"
    
    # Everything else is normal
    return "normal"


# Test it
test_data = [
    ("Alice",  "Hello everyone! Good morning 😊"),
    ("Bob",    "<Media omitted>"),
    ("Charlie","This message was deleted"),
    ("System", "Messages and calls are end-to-end encrypted."),
    ("System", "Dave joined using this group's invite link"),
    ("Alice",  "Check https://github.com/test"),
    ("Dave",   "😂😂😂"),
]

print(f"{'SENDER':<12} {'MESSAGE':<40} {'TYPE'}")
print("-" * 70)

for sender, message in test_data:
    msg_type = get_message_type(sender, message)
    print(f"{sender:<12} {message[:38]:<40} {msg_type}")

SENDER       MESSAGE                                  TYPE
----------------------------------------------------------------------
Alice        Hello everyone! Good morning 😊           normal
Bob          <Media omitted>                          media
Charlie      This message was deleted                 deleted
System       Messages and calls are end-to-end encr   system
System       Dave joined using this group's invite    group_event
Alice        Check https://github.com/test            normal
Dave         😂😂😂                                      normal


In [7]:
# ============================================
# CELL 9: THE COMPLETE PARSER
# Everything we learned, combined into one
# clean function
# ============================================

import re
import pandas as pd

def parse_whatsapp_chat(file_path):
    """
    Parse a WhatsApp exported chat file into a DataFrame.
    
    Parameters:
        file_path (str): Path to the .txt file
    
    Returns:
        pd.DataFrame with columns:
        [datetime_str, date, time, sender, message, message_type]
    """
    
    # Our timestamp pattern
    TIMESTAMP_PATTERN = r"(\d{1,2}/\d{1,2}/\d{2,4}),\s(\d{1,2}:\d{2}\s[AaPp][Mm])\s-\s(.*)"
    
    # Read all lines
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    # Storage
    messages = []
    
    # Temporary holders for current message being built
    current_date    = None
    current_time    = None
    current_sender  = None
    current_message = None
    
    for line in lines:
        line = line.strip()  # Remove \n from end
        
        if not line:          # Skip empty lines
            continue
        
        match = re.match(TIMESTAMP_PATTERN, line)
        
        if match:
            # ── This is a NEW message ──────────────────
            
            # Save the PREVIOUS message before starting new one
            if current_message is not None:
                messages.append({
                    "date"        : current_date,
                    "time"        : current_time,
                    "sender"      : current_sender,
                    "message"     : current_message,
                    "message_type": get_message_type(current_sender, 
                                                     current_message)
                })
            
            # Extract parts of new message
            current_date = match.group(1)
            current_time = match.group(2)
            content      = match.group(3)
            
            # Split sender from message
            parts = content.split(":", 1)
            if len(parts) == 2:
                current_sender  = parts[0].strip()
                current_message = parts[1].strip()
            else:
                current_sender  = "System"
                current_message = content.strip()
        
        else:
            # ── This is a CONTINUATION line ────────────
            if current_message is not None:
                # Append to current message with a space
                current_message = current_message + " " + line
    
    # ── Save the LAST message ──────────────────────────
    # The loop ends without saving the final message
    if current_message is not None:
        messages.append({
            "date"        : current_date,
            "time"        : current_time,
            "sender"      : current_sender,
            "message"     : current_message,
            "message_type": get_message_type(current_sender, current_message)
        })
    
    # Convert to DataFrame
    df = pd.DataFrame(messages)
    
    return df


# ── RUN THE PARSER ─────────────────────────────────────
file_path = "../data/sample/sample_chat.txt"
df = parse_whatsapp_chat(file_path)

print(f"✅ Parsing complete!")
print(f"   Total messages parsed : {len(df)}")
print(f"   Columns created       : {list(df.columns)}")
print(f"   Unique senders        : {df['sender'].unique()}")
print(f"\nMessage type breakdown:")
print(df['message_type'].value_counts())

✅ Parsing complete!
   Total messages parsed : 25
   Columns created       : ['date', 'time', 'sender', 'message', 'message_type']
   Unique senders        : ['System' 'Alice' 'Bob' 'Charlie' 'Dave']

Message type breakdown:
message_type
normal         19
media           3
system          1
deleted         1
group_event     1
Name: count, dtype: int64


In [8]:
# ============================================
# CELL 10: Look at what we built
# ============================================

print("FIRST 5 ROWS:")
print("=" * 55)
print(df.head())

print(f"\nFULL DATAFRAME:")
print("=" * 55)

# Print each row clearly
for i, row in df.iterrows():
    print(f"\n[{i+1}] {row['date']} {row['time']}")
    print(f"     Sender  : {row['sender']}")
    print(f"     Type    : {row['message_type']}")
    print(f"     Message : {row['message'][:50]}")

FIRST 5 ROWS:
      date     time  sender  \
0  3/14/23  9:42 AM  System   
1  3/14/23  9:43 AM   Alice   
2  3/14/23  9:44 AM     Bob   
3  3/14/23  9:44 AM   Alice   
4  3/14/23  9:45 AM     Bob   

                                             message message_type  
0  Messages and calls are end-to-end encrypted. N...       system  
1                       Hey everyone! Good morning 😊       normal  
2                              Morning! How are you?       normal  
3       I'm doing great! Just finished my assignment       normal  
4                                    <Media omitted>        media  

FULL DATAFRAME:

[1] 3/14/23 9:42 AM
     Sender  : System
     Type    : system
     Message : Messages and calls are end-to-end encrypted. No on

[2] 3/14/23 9:43 AM
     Sender  : Alice
     Type    : normal
     Message : Hey everyone! Good morning 😊

[3] 3/14/23 9:44 AM
     Sender  : Bob
     Type    : normal
     Message : Morning! How are you?

[4] 3/14/23 9:44 AM
     Sender  : 

In [9]:
# ============================================
# CELL 11: Parser Validation
# Verify every row is correct
# ============================================

print("PARSER VALIDATION REPORT")
print("=" * 55)

# Check 1: Total messages
print(f"\n✅ Total messages     : {len(df)}")

# Check 2: Multi-line merge
alice_friday = df[df['message'].str.contains('Friday', na=False)]
print(f"\n✅ Multi-line merge check:")
for _, row in alice_friday.iterrows():
    print(f"   Message : {row['message']}")
    has_continuation = "more time thankfully" in row['message']
    print(f"   Contains continuation: {has_continuation}")

# Check 3: Message types
print(f"\n✅ Message type breakdown:")
print(df['message_type'].value_counts().to_string())

# Check 4: No None values in critical columns
print(f"\n✅ Missing values check:")
print(df[['date','time','sender','message']].isnull().sum().to_string())

# Check 5: All senders found
print(f"\n✅ All senders found:")
for sender in df['sender'].unique():
    count = len(df[df['sender'] == sender])
    print(f"   {sender:<12} → {count} messages")

# Check 6: Show the merged multi-line message fully
print(f"\n✅ Full multi-line message:")
msg = df[df['message'].str.contains('Friday', na=False)]['message'].values[0]
print(f"   '{msg}'")

PARSER VALIDATION REPORT

✅ Total messages     : 25

✅ Multi-line merge check:
   Message : It's been moved to next Friday which gives us more time thankfully
   Contains continuation: True

✅ Message type breakdown:
message_type
normal         19
media           3
system          1
deleted         1
group_event     1

✅ Missing values check:
date       0
time       0
sender     0
message    0

✅ All senders found:
   System       → 2 messages
   Alice        → 8 messages
   Bob          → 6 messages
   Charlie      → 5 messages
   Dave         → 4 messages

✅ Full multi-line message:
   'It's been moved to next Friday which gives us more time thankfully'


In [10]:
# ============================================
# CELL 13: Understanding what features we need
# and WHY each one matters for analysis
# ============================================

import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))
from src.parser import parse_whatsapp_chat

df = parse_whatsapp_chat("../data/sample/sample_chat.txt")

print("CURRENT DataFrame - what we HAVE:")
print("=" * 55)
print(df.dtypes)
print()
print(df[['date','time','message']].head(3))

print()
print("PROBLEM 1: date and time are strings")
print(f"  date column type : {type(df['date'][0])}")
print(f"  Example value    : {df['date'][0]}")
print(f"  Can we do math?  : NO - it's just text")

print()
print("PROBLEM 2: message has hidden information")
print(f"  Example message  : {df['message'][1]}")
print(f"  Word count       : ???  (not extracted yet)")
print(f"  Emoji count      : ???  (not extracted yet)")
print(f"  Has URL          : ???  (not extracted yet)")

print()
print("SOLUTION: Feature Engineering in preprocessor.py")
print("We extract all hidden information into new columns")

CURRENT DataFrame - what we HAVE:
date            object
time            object
sender          object
message         object
message_type    object
dtype: object

      date     time                                            message
0  3/14/23  9:42 AM  Messages and calls are end-to-end encrypted. N...
1  3/14/23  9:43 AM                       Hey everyone! Good morning 😊
2  3/14/23  9:44 AM                              Morning! How are you?

PROBLEM 1: date and time are strings
  date column type : <class 'str'>
  Example value    : 3/14/23
  Can we do math?  : NO - it's just text

PROBLEM 2: message has hidden information
  Example message  : Hey everyone! Good morning 😊
  Word count       : ???  (not extracted yet)
  Emoji count      : ???  (not extracted yet)
  Has URL          : ???  (not extracted yet)

SOLUTION: Feature Engineering in preprocessor.py
We extract all hidden information into new columns


In [11]:
# ============================================
# CELL 14: Extract features from date and time
# 
# Key concept: pd.to_datetime()
# Converts string "3/14/23" into a real 
# datetime object Python can do math with
# ============================================

import pandas as pd

# Take our parsed df
# Combine date and time into one string first
# "3/14/23" + " " + "9:42 AM" = "3/14/23 9:42 AM"
df['datetime_str'] = df['date'] + " " + df['time']

print("Combined datetime string:")
print(df['datetime_str'].head(3))
print(f"Type: {type(df['datetime_str'][0])}")

print()

# Convert string to actual datetime object
df['datetime'] = pd.to_datetime(df['datetime_str'], 
                                 format="%m/%d/%y %I:%M %p")

print("After pd.to_datetime():")
print(df['datetime'].head(3))
print(f"Type: {type(df['datetime'][0])}")

print()

# NOW we can extract individual components
df['year']    = df['datetime'].dt.year
df['month']   = df['datetime'].dt.month
df['day']     = df['datetime'].dt.day
df['hour']    = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.day_name()

print("Extracted date features:")
print(df[['datetime','year','month','day',
          'hour','weekday']].head(5))

Combined datetime string:
0    3/14/23 9:42 AM
1    3/14/23 9:43 AM
2    3/14/23 9:44 AM
Name: datetime_str, dtype: object
Type: <class 'str'>

After pd.to_datetime():
0   2023-03-14 09:42:00
1   2023-03-14 09:43:00
2   2023-03-14 09:44:00
Name: datetime, dtype: datetime64[ns]
Type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>

Extracted date features:
             datetime  year  month  day  hour  weekday
0 2023-03-14 09:42:00  2023      3   14     9  Tuesday
1 2023-03-14 09:43:00  2023      3   14     9  Tuesday
2 2023-03-14 09:44:00  2023      3   14     9  Tuesday
3 2023-03-14 09:44:00  2023      3   14     9  Tuesday
4 2023-03-14 09:45:00  2023      3   14     9  Tuesday


In [12]:
# ============================================
# CELL 15: Extract features from message text
# ============================================

import emoji
import re

# Work only on normal messages for text features
# Media/deleted messages have no real text to analyze

def count_emojis(text):
    """Count number of emojis in text"""
    return len([c for c in text if c in emoji.EMOJI_DATA])

def count_words(text):
    """Count words - split on whitespace"""
    return len(str(text).split())

def count_urls(text):
    """Count URLs in message"""
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    return len(re.findall(url_pattern, str(text)))

def has_url(text):
    """Check if message contains a URL"""
    return count_urls(text) > 0


# Apply to dataframe
df['message_length'] = df['message'].apply(len)
df['word_count']     = df['message'].apply(count_words)
df['emoji_count']    = df['message'].apply(count_emojis)
df['url_count']      = df['message'].apply(count_urls)
df['has_url']        = df['message'].apply(has_url)
df['is_media']       = df['message_type'] == 'media'
df['is_deleted']     = df['message_type'] == 'deleted'

print("Message features extracted!")
print("=" * 55)
print(df[['sender','message','message_length',
          'word_count','emoji_count','has_url']].head(8))

print()
print("Emoji analysis:")
emoji_df = df[df['emoji_count'] > 0][['sender','message','emoji_count']]
print(emoji_df)

Message features extracted!
    sender                                            message  message_length  \
0   System  Messages and calls are end-to-end encrypted. N...             139   
1    Alice                       Hey everyone! Good morning 😊              28   
2      Bob                              Morning! How are you?              21   
3    Alice       I'm doing great! Just finished my assignment              44   
4      Bob                                    <Media omitted>              15   
5  Charlie                           This message was deleted              24   
6    Alice              Did you all see the project deadline?              37   
7    Alice  It's been moved to next Friday which gives us ...              66   

   word_count  emoji_count  has_url  
0          25            0    False  
1           5            1    False  
2           4            0    False  
3           7            0    False  
4           2            0    False  
5           4 

In [13]:
# ============================================
# CELL 16: See the complete enriched DataFrame
# ============================================

print("COMPLETE FEATURE LIST:")
print("=" * 55)
for col in df.columns:
    print(f"  {col:<20} → {df[col].dtype} "
          f"| example: {df[col][1]}")

print()
print(f"DataFrame shape: {df.shape}")
print(f"Rows    : {df.shape[0]} messages")
print(f"Columns : {df.shape[1]} features")

print()
print("BEFORE feature engineering: 5 columns")
print("AFTER  feature engineering:", df.shape[1], "columns")
print()
print("These extra columns power ALL your analysis!")

COMPLETE FEATURE LIST:
  date                 → object | example: 3/14/23
  time                 → object | example: 9:43 AM
  sender               → object | example: Alice
  message              → object | example: Hey everyone! Good morning 😊
  message_type         → object | example: normal
  datetime_str         → object | example: 3/14/23 9:43 AM
  datetime             → datetime64[ns] | example: 2023-03-14 09:43:00
  year                 → int32 | example: 2023
  month                → int32 | example: 3
  day                  → int32 | example: 14
  hour                 → int32 | example: 9
  weekday              → object | example: Tuesday
  message_length       → int64 | example: 28
  word_count           → int64 | example: 5
  emoji_count          → int64 | example: 1
  url_count            → int64 | example: 0
  has_url              → bool | example: False
  is_media             → bool | example: False
  is_deleted           → bool | example: False

DataFrame shape: (25, 19

In [15]:
# ============================================
# CELL 17: Test the complete pipeline
# Parser → Preprocessor → Ready for Analysis
# ============================================

import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))

from src.parser       import parse_whatsapp_chat
from src.preprocessor import preprocess

# Step 1: Parse
df_raw = parse_whatsapp_chat("../data/sample/sample_chat.txt")
print(f"After parsing    : {df_raw.shape[1]} columns")

# Step 2: Preprocess  
df = preprocess(df_raw)
print(f"After preprocess : {df.shape[1]} columns")

print()
print("NEW COLUMNS ADDED:")
new_cols = [c for c in df.columns if c not in df_raw.columns]
for col in new_cols:
    print(f"  ✅ {col}")

print()
print("SAMPLE DATA:")
print(df[['sender','hour','weekday',
          'word_count','emoji_count','has_url']].head(8))

print()
print("QUICK INSIGHTS:")
print(f"  Most active hour  : {df['hour'].mode()[0]}:00")
print(f"  Most active day   : {df['weekday'].mode()[0]}")
print(f"  Avg word count    : {df['word_count'].mean():.1f}")
print(f"  Total emojis sent : {df['emoji_count'].sum()}")
print(f"  Messages with URL : {df['has_url'].sum()}")

After parsing    : 5 columns
After preprocess : 19 columns

NEW COLUMNS ADDED:
  ✅ datetime
  ✅ year
  ✅ month
  ✅ day
  ✅ hour
  ✅ weekday
  ✅ date_only
  ✅ message_length
  ✅ word_count
  ✅ emoji_count
  ✅ url_count
  ✅ has_url
  ✅ is_media
  ✅ is_deleted

SAMPLE DATA:
    sender  hour  weekday  word_count  emoji_count  has_url
0   System     9  Tuesday           0            0    False
1    Alice     9  Tuesday           5            1    False
2      Bob     9  Tuesday           4            0    False
3    Alice     9  Tuesday           7            0    False
4      Bob     9  Tuesday           0            0    False
5  Charlie     9  Tuesday           0            0    False
6    Alice     9  Tuesday           7            0    False
7    Alice     9  Tuesday          12            0    False

QUICK INSIGHTS:
  Most active hour  : 9:00
  Most active day   : Tuesday
  Avg word count    : 3.3
  Total emojis sent : 13
  Messages with URL : 1
